# LECTURE 17 — DEMONSTRATION
### MODULE 7: UNSUPERVISED LEARNING & DIMENSIONALITY REDUCTION

**DEMONSTRATION** — Clustering African Economies Using Vulnerability Indicators

*WAIFEM ML Facilitation Programme 2026 — Compatible with Google Colab & VS Code*

## ── SECTION 1: IMPORTS

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples

from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 11})

try:
    import google.colab          # noqa: F401
    print("Environment : Google Colab")
except ImportError:
    print("Environment : Local (VS Code / terminal)")

print("Libraries loaded successfully.\n")

## ── SECTION 2: AFRICAN ECONOMIES DATA

In [ ]:
COUNTRIES = [
    'Nigeria',        'South Africa',   'Egypt',          'Kenya',
    'Ethiopia',       'Ghana',          'Tanzania',       "Cote d'Ivoire",
    'Cameroon',       'Angola',         'Uganda',         'Mozambique',
    'Zambia',         'Zimbabwe',       'Senegal',        'Mali',
    'Burkina Faso',   'Guinea',         'Niger',          'Rwanda',
    'Benin',          'Burundi',        'Sierra Leone',   'Togo',
    'Libya',          'Tunisia',        'Algeria',        'Morocco',
    'Sudan',          'Madagascar',     'Botswana',       'Namibia',
    'Mauritius',      'Malawi',         'DR Congo',       'Congo Republic',
    'Gabon',          'Eq. Guinea',     'Eswatini',       'Lesotho',
    'Djibouti',       'Eritrea',        'Somalia',        'Chad',
    'Cabo Verde',
]
N = len(COUNTRIES)   # 45 countries


def generate_vulnerability_data(countries: list, seed: int = 42) -> pd.DataFrame:
    """
    Synthesise annual vulnerability indicators for 45 African economies.
    Countries are seeded into three latent clusters:
      A — Low vulnerability (diversified / middle-income)
      B — Moderate vulnerability (commodity-dependent)
      C — High vulnerability (fragile / low-income)
    """
    rng = np.random.default_rng(seed)

    # Cluster assignments (A=0, B=1, C=2)
    cluster_true = np.array(
        [0]*14 +   # first 14 → Group A
        [1]*16 +   # next  16 → Group B
        [2]*15     # last  15 → Group C
    )

    # Cluster centres for each indicator
    #              gdp_pc  infl  debt  ca_bal  reserves  fiscal  unempl  credit
    centres = np.array([
        [3500,   5,   45,  -2,   5.5,  -1.5,   8,   45],    # A  low vuln
        [1200,  12,   60,  -4,   3.0,  -3.5,  12,   25],    # B  mod vuln
        [ 600,  20,   80,  -6,   1.5,  -6.0,  18,   12],    # C  high vuln
    ], dtype=float)

    # Within-cluster noise scales
    noise = np.array([400, 3, 10, 1.5, 0.8, 1.0, 3, 8], dtype=float)

    X = np.zeros((len(countries), 8))
    for i, c in enumerate(cluster_true):
        X[i] = centres[c] + rng.normal(0, noise)

    # Clip to realistic ranges
    X[:, 0] = np.clip(X[:, 0], 300,   8000)   # gdp_pc
    X[:, 1] = np.clip(X[:, 1],   2,     45)   # inflation
    X[:, 2] = np.clip(X[:, 2],  20,    120)   # debt/GDP
    X[:, 3] = np.clip(X[:, 3], -15,      5)   # current account
    X[:, 4] = np.clip(X[:, 4],   0.5,    9)   # reserves (months)
    X[:, 5] = np.clip(X[:, 5], -12,      2)   # fiscal balance
    X[:, 6] = np.clip(X[:, 6],   3,     35)   # unemployment
    X[:, 7] = np.clip(X[:, 7],   5,     80)   # credit/GDP

    cols = ['gdp_per_capita', 'inflation', 'debt_to_gdp',
            'current_account', 'reserves_months', 'fiscal_balance',
            'unemployment', 'credit_to_gdp']

    df = pd.DataFrame(X, index=countries, columns=cols)
    df['true_cluster'] = cluster_true
    return df


df = generate_vulnerability_data(COUNTRIES)
INDICATOR_COLS = [c for c in df.columns if c != 'true_cluster']

print(f"Dataset : {df.shape[0]} African economies  x  {len(INDICATOR_COLS)} indicators")
print(df.describe().round(2).to_string())

## ── SECTION 3: STANDARDISE FEATURES

In [ ]:
scaler = StandardScaler()
X_sc   = scaler.fit_transform(df[INDICATOR_COLS].values)
X_df   = pd.DataFrame(X_sc, index=df.index, columns=INDICATOR_COLS)

print("\nFeatures standardised (mean=0, std=1).")

## ── SECTION 4: K-MEANS — ELBOW METHOD

In [ ]:
print("\nRunning K-Means elbow analysis (k = 2 to 10) ...")

inertias    = []
sil_scores  = []
K_RANGE     = range(2, 11)

for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    km.fit(X_sc)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_sc, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('K-Means Cluster Validation', fontsize=12, fontweight='bold')

axes[0].plot(list(K_RANGE), inertias, marker='o', color='steelblue', linewidth=1.8)
axes[0].set_title('Elbow Method (Within-Cluster Inertia)')
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia')
axes[0].grid(True, alpha=0.3)

axes[1].plot(list(K_RANGE), sil_scores, marker='s', color='seagreen', linewidth=1.8)
axes[1].set_title('Silhouette Score')
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].grid(True, alpha=0.3)

optimal_k = list(K_RANGE)[int(np.argmax(sil_scores))]
axes[1].axvline(optimal_k, color='tomato', linestyle='--', linewidth=1.2,
                label=f'Optimal k = {optimal_k}')
axes[1].legend()

plt.tight_layout()
plt.savefig('lecture17_demo_elbow_silhouette.png', bbox_inches='tight')
plt.show()
print(f"Optimal k (highest silhouette) : {optimal_k}")
print("Saved : lecture17_demo_elbow_silhouette.png")

## ── SECTION 5: K-MEANS FINAL CLUSTERING

In [ ]:
K_FINAL = 3    # use 3 (matches economic intuition: low/medium/high vulnerability)

kmeans = KMeans(n_clusters=K_FINAL, random_state=SEED, n_init=20)
df['kmeans_cluster'] = kmeans.fit_predict(X_sc)

sil_final = silhouette_score(X_sc, df['kmeans_cluster'])
print(f"\nK-Means (k={K_FINAL}) — Silhouette Score : {sil_final:.4f}")

# Cluster profile table
print("\nCluster Profiles (mean values per cluster):")
profile = df.groupby('kmeans_cluster')[INDICATOR_COLS].mean().round(2)
print(profile.to_string())

## ── SECTION 6: SILHOUETTE PLOT

In [ ]:
sil_vals    = silhouette_samples(X_sc, df['kmeans_cluster'])
palette     = sns.color_palette('Set2', K_FINAL)

fig, ax = plt.subplots(figsize=(9, 6))
y_lower = 10
for k in range(K_FINAL):
    vals_k = np.sort(sil_vals[df['kmeans_cluster'] == k])
    size_k = len(vals_k)
    y_upper = y_lower + size_k
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, vals_k,
                     facecolor=palette[k], alpha=0.7, label=f'Cluster {k}')
    ax.text(-0.05, y_lower + 0.5 * size_k, str(k))
    y_lower = y_upper + 10

ax.axvline(sil_final, color='tomato', linestyle='--', linewidth=1.5,
           label=f'Mean silhouette = {sil_final:.3f}')
ax.set_title(f'Silhouette Plot — K-Means (k={K_FINAL})', fontweight='bold')
ax.set_xlabel('Silhouette Coefficient')
ax.set_ylabel('Cluster')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('lecture17_demo_silhouette_plot.png', bbox_inches='tight')
plt.show()
print("Saved : lecture17_demo_silhouette_plot.png")

## ── SECTION 7: HIERARCHICAL CLUSTERING & DENDROGRAM

In [ ]:
print("\nBuilding hierarchical clustering dendrogram (Ward linkage) ...")

linkage_matrix = linkage(X_sc, method='ward', metric='euclidean')

fig, ax = plt.subplots(figsize=(16, 6))
dend = dendrogram(
    linkage_matrix,
    labels=df.index.tolist(),
    leaf_rotation=90,
    leaf_font_size=7,
    color_threshold=0.6 * max(linkage_matrix[:, 2]),
    ax=ax,
)
ax.set_title('Hierarchical Clustering Dendrogram — African Economies (Ward Linkage)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Country')
ax.set_ylabel('Distance')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('lecture17_demo_dendrogram.png', bbox_inches='tight')
plt.show()
print("Saved : lecture17_demo_dendrogram.png")

# Cut tree into 3 clusters
agg = AgglomerativeClustering(n_clusters=K_FINAL, linkage='ward')
df['hier_cluster'] = agg.fit_predict(X_sc)

## ── SECTION 8: 2-D VISUALISATION (PCA PROJECTION)

In [ ]:
pca2 = PCA(n_components=2, random_state=SEED)
coords = pca2.fit_transform(X_sc)

var_pc1 = pca2.explained_variance_ratio_[0] * 100
var_pc2 = pca2.explained_variance_ratio_[1] * 100

cluster_labels = {0: 'Low Vulnerability', 1: 'Moderate Vulnerability', 2: 'High Vulnerability'}
color_map      = {0: '#2196F3', 1: '#FF9800', 2: '#F44336'}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('African Economies — Cluster Visualisation (PCA 2-D Projection)',
             fontsize=12, fontweight='bold')

for ax, cluster_col, title in zip(
    axes,
    ['kmeans_cluster', 'hier_cluster'],
    [f'K-Means (k={K_FINAL})', f'Hierarchical — Ward (k={K_FINAL})'],
):
    for k in range(K_FINAL):
        mask = df[cluster_col] == k
        ax.scatter(coords[mask, 0], coords[mask, 1],
                   c=color_map[k], label=cluster_labels[k],
                   s=80, alpha=0.85, edgecolors='white', linewidths=0.5)
        for i, name in enumerate(df.index[mask]):
            ax.annotate(name, (coords[mask, 0][i], coords[mask, 1][i]),
                        fontsize=5.5, ha='center', va='bottom',
                        xytext=(0, 4), textcoords='offset points')

    ax.set_title(title)
    ax.set_xlabel(f'PC1 ({var_pc1:.1f} % variance)')
    ax.set_ylabel(f'PC2 ({var_pc2:.1f} % variance)')
    ax.legend(fontsize=8, loc='best')
    ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.savefig('lecture17_demo_cluster_scatter.png', bbox_inches='tight')
plt.show()
print("Saved : lecture17_demo_cluster_scatter.png")

## ── SECTION 9: CLUSTER PROFILES (HEATMAP)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
profile_z = X_df.groupby(df['kmeans_cluster']).mean()
profile_z.index = [cluster_labels[i] for i in profile_z.index]

sns.heatmap(profile_z, annot=True, fmt='.2f', cmap='RdYlGn_r',
            linewidths=0.5, ax=ax, center=0)
ax.set_title(f'K-Means Cluster Profiles — Standardised Indicator Means  (k={K_FINAL})',
             fontweight='bold')
ax.set_xlabel('Indicator')
ax.set_ylabel('Cluster')
plt.tight_layout()
plt.savefig('lecture17_demo_cluster_heatmap.png', bbox_inches='tight')
plt.show()
print("Saved : lecture17_demo_cluster_heatmap.png")

# Policy narrative
print("""
── Policy Interpretation of Clusters ───────────────────────────────────────
  Cluster 0 — Low Vulnerability
    Higher GDP per capita, lower inflation and debt, stronger reserves.
    Policy priority: maintain macro stability, expand credit access.

  Cluster 1 — Moderate Vulnerability
    Mid-range indicators, commodity-dependent growth patterns.
    Policy priority: diversification, fiscal consolidation.

  Cluster 2 — High Vulnerability
    Low income, elevated inflation and debt, thin reserves.
    Policy priority: debt relief, external support, stabilisation.
────────────────────────────────────────────────────────────────────────────
""")

## ── SECTION 10: SUMMARY

In [ ]:
print("""
╔═══════════════════════════════════════════════════════════════════╗
║  LECTURE 17 DEMO COMPLETE                                         ║
╠═══════════════════════════════════════════════════════════════════╣
║  Concepts demonstrated:                                           ║
║   1. Synthetic vulnerability indicators for 45 African economies  ║
║   2. StandardScaler normalisation                                 ║
║   3. K-Means — elbow method & silhouette score for optimal k      ║
║   4. Silhouette plot for within-cluster cohesion                  ║
║   5. Hierarchical clustering — Ward linkage + dendrogram          ║
║   6. PCA 2-D projection for cluster visualisation                 ║
║   7. Cluster profile heatmap for policy interpretation            ║
╠═══════════════════════════════════════════════════════════════════╣
║  Saved outputs:                                                   ║
║   lecture17_demo_elbow_silhouette.png                             ║
║   lecture17_demo_silhouette_plot.png                              ║
║   lecture17_demo_dendrogram.png                                   ║
║   lecture17_demo_cluster_scatter.png                              ║
║   lecture17_demo_cluster_heatmap.png                              ║
╚═══════════════════════════════════════════════════════════════════╝
""")